In [1]:
!pip install sentence_transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 107.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitli

In [2]:
# Core Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
from scipy.stats import randint, uniform

In [3]:
from sentence_transformers import SentenceTransformer

In [22]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F

## IMPORT & EXPLORE

In [19]:
labse_transformer = SentenceTransformer('sentence-transformers/LaBSE')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

2_Dense/pytorch_model.bin:   0%|          | 0.00/2.36M [00:00<?, ?B/s]

In [5]:
all_products_combined = pd.read_csv(r"/content/flip_data_vlm/flip_data_vlm/all_products_combined.csv")

## CLEAN & PREPARE

In [4]:
!unzip -q "/content/drive/MyDrive/Data Science/flip_vlm/flip_data_vlm.zip" -d /content/flip_data_vlm

In [45]:
all_products_combined['colab_image_path'] = (
    "/content/flip_data_vlm/flip_data_vlm/" +
    all_products_combined['local_image_path']
        .fillna("")                               # replace NaN with empty string
        .astype(str)                              # ensure everything is string
        .apply(lambda x: x.split("flip/data/")[-1])
)


In [46]:
all_products_combined[['colab_image_path','title']].sample(10)

,colab_image_path,title
10308,/content/flip_data_vlm/flip_data_vlm/category_...,Фигурка «Конь-качалка»
8273,/content/flip_data_vlm/flip_data_vlm/category_...,"Менажница, молочный"
5683,/content/flip_data_vlm/flip_data_vlm/category_...,Нож охотничий «Ролло»
16198,/content/flip_data_vlm/flip_data_vlm/category_...,Тостер
303,/content/flip_data_vlm/flip_data_vlm/category_...,Протеиновые брауни в шоколаде со вкусом вишни
15509,/content/flip_data_vlm/flip_data_vlm/category_...,Выпрямитель для волос VT-2317
11841,/content/flip_data_vlm/flip_data_vlm/category_...,Свеча для торта «Цифра 8»
1299,/content/flip_data_vlm/flip_data_vlm/category_...,Коробка для мелочей двухсторонняя
3949,/content/flip_data_vlm/flip_data_vlm/category_...,"Бутылка для воды «Без эспрессо, я в депрессо»"
5408,/content/flip_data_vlm/flip_data_vlm/category_...,"Термокружка «Планы на день», черный"


In [48]:
image_title_pairs_sampled = list(all_products_combined[['colab_image_path','title']].sample(5000, random_state = 42).itertuples(index=False, name=None))

In [49]:
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T

class ImageTextDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data  # list of (image_path, text)
        self.transform = transform or T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406],
                            [0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, text = self.data[idx]
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        return image, text

In [50]:
dataset = ImageTextDataset(image_title_pairs_sampled)  # your (image, text) tuples
dataloader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)

## MODEL BUILD

In [23]:
class FlipDualEncoder(nn.Module):
  def __init__(self, embedding_dim = 256):
    super(FlipDualEncoder, self).__init__()

        #ResNet50 as visual encoder
        resnet50 = models.resnet50(pretrained = True)
        resnet50.fc = nn.Identity()
        self.visual_encoder = resnet50
        self.image_projection = nn.Linear(2048, embedding_dim)
    
        #LaBSE as text encoder
        self.text_encoder = SentenceTransformer('sentence-transformers/LaBSE')
        self.text_projection = nn.Linear(768, embedding_dim)

  def forward(self, images, texts):

    # Image embedding
    image_encoded = self.visual_encoder(images)
    image_embedding = F.normalize(self.image_projection(image_encoded), p=2, dim=-1)

    # Text embedding
    with torch.no_grad():  # freeze LaBSE initially (optional)
        text_features = self.text_encoder.encode(
            texts,
            convert_to_tensor=True,
            normalize_embeddings=False
        )  # (B, 768)

    text_embedding = self.text_projection(text_features)  # (B, 256)
    text_embedding = nn.functional.normalize(text_embedding, p=2, dim=-1)

    return image_embedding, text_embedding

In [25]:
def cosine_similarity_loss(image_embeddings, text_embeddings, temperature=0.07):
    """
    image_embeddings: (B, D)
    text_embeddings:  (B, D)
    """
    # Cosine similarity matrix: (B, B)
    similarity_matrix = torch.matmul(image_embeddings, text_embeddings.T) / temperature

    # Labels: diagonal is positive
    targets = torch.arange(similarity_matrix.size(0)).to(similarity_matrix.device)

    # Symmetric loss: image-to-text + text-to-image
    loss_i2t = F.cross_entropy(similarity_matrix, targets)
    loss_t2i = F.cross_entropy(similarity_matrix.T, targets)

    return (loss_i2t + loss_t2i) / 2

## TRAINING

In [52]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [54]:
flip_resnet_sentence_encoder = FlipDualEncoder(embedding_dim=256).to(device)
optimizer = torch.optim.Adam(flip_resnet_sentence_encoder.parameters(), lr=1e-4)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [55]:
for epoch in range(100):
    print(f"Epoch {epoch+1} started")

    for step, (images, texts) in enumerate(dataloader):
        print(f"  Step {step} | Batch size: {len(images)}")

        images = images.to(device)
        image_emb, text_emb = flip_resnet_sentence_encoder(images, texts)
        loss = cosine_similarity_loss(image_emb, text_emb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"✅ Epoch {epoch+1} finished | Loss: {loss.item():.4f}")

Streaming output truncated to the last 5000 lines.
  Step 87 | Batch size: 32
  Step 88 | Batch size: 32
  Step 89 | Batch size: 32
  Step 90 | Batch size: 32
  Step 91 | Batch size: 32
  Step 92 | Batch size: 32
  Step 93 | Batch size: 32
  Step 94 | Batch size: 32
  Step 95 | Batch size: 32
  Step 96 | Batch size: 32
  Step 97 | Batch size: 32
  Step 98 | Batch size: 32
  Step 99 | Batch size: 32
  Step 100 | Batch size: 32
  Step 101 | Batch size: 32
  Step 102 | Batch size: 32
  Step 103 | Batch size: 32
  Step 104 | Batch size: 32
  Step 105 | Batch size: 32
  Step 106 | Batch size: 32
  Step 107 | Batch size: 32
  Step 108 | Batch size: 32
  Step 109 | Batch size: 32
  Step 110 | Batch size: 32
  Step 111 | Batch size: 32
  Step 112 | Batch size: 32
  Step 113 | Batch size: 32
  Step 114 | Batch size: 32
  Step 115 | Batch size: 32
  Step 116 | Batch size: 32
  Step 117 | Batch size: 32
  Step 118 | Batch size: 32
  Step 119 | Batch size: 32
  Step 120 | Batch size: 32
  Step 121

In [56]:
torch.save(
    flip_resnet_sentence_encoder.state_dict(),
    f"/content/drive/MyDrive/Data Science/flip_vlm/flip_dual_encoder5000.pt"
)

In [80]:
total_params = sum(p.numel() for p in flip_resnet_sentence_encoder.parameters())
trainable_params = sum(p.numel() for p in flip_resnet_sentence_encoder.parameters() if p.requires_grad)

print(f"🧠 Total Parameters: {total_params:,}")
print(f"✅ Trainable Parameters: {trainable_params:,}")

🧠 Total Parameters: 495,746,880
✅ Trainable Parameters: 495,746,880


In [58]:
!ls -lh "/content/drive/MyDrive/Data Science/flip_vlm"

total 3.6G
-rw------- 1 root root 1.7G Jul  9 06:09 flip_data_vlm.zip
-rw------- 1 root root 1.9G Jul  9 09:11 flip_dual_encoder5000.pt
-rw------- 1 root root 308K Jul  9 09:12 flip_vlm_training.ipynb


## EVALUATION

In [60]:
flip_resnet_sentence_encoder.eval()

FlipDualEncoder(
  (visual_encoder): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequent

In [63]:
from torch.nn.functional import cosine_similarity